# 08 · AIA 72-hour loader, training-step and checkpoint check

## tl;dr
This is a **technical check, not the full AIA experiment**. It loads the first and last eligible cases in each of the seven frozen roles: 14 cases and 42 images. It tests both image metadata formats, trains fresh weights for two steps on the two training-role cases, and verifies exact replay after saving and reloading. Observed results appear below.

The checkpoint and normalization from this notebook must **not** initialize the scientific experiment. No forecast-skill, fusion improvement or uncertainty coverage is measured here.

## Context & Methods
We reuse Bamidele's temporal CNN–GRU architecture with a fresh 72-hour target and the roles from Notebook 07. Images contain upstream processed pixel values; they are not raw physical radiances. The manifest supplies the 72-hour outcome. Embedded 48-hour labels are ignored.

### Key assumptions
- Selection uses issue times, not outcome classes. Two boundary cases per role are deliberately a software check, not a representative evaluation sample.
- Each frame's pinned generation, SHA-256, embedded identity, channel order and finite values are checked. Nominal history times precede issue time. Actual channel observation times and historical delivery are not certified.
- Six channels are transformed with training-only asinh scaling and standardization, then resized from 512 to 256 pixels using bilinear interpolation. Only the two technical training cases fit these temporary statistics.
- All other roles are forward-pass checks only. No label-based accuracy is computed.
- This local CPU notebook requires the already staged small image sample. It does not start cloud resources or change other jobs.

### 1. Configuration and pinned inputs
Run inside the repository with `requirements-training-lock.txt`. The bounded acquisition helper is `scripts/stage_aia72_canary.py`; its saved plan and receipt identify the input objects. Reruns use a new output directory.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from IPython.display import display, Markdown

root = Path.cwd().resolve()
if not (root / 'scripts/aia72_data.py').is_file():
    root = root.parent
assert (root / 'scripts/aia72_data.py').is_file(), 'Run inside the Gray-Box repository'
sys.path.insert(0, str(root))
from scripts.aia72_data import AIA72Dataset, LAGS, fit_normalization, transform_image, validate_frame_identity
from scripts.aia_io import CHANNELS, load_aia_frame
from scripts.dataset_io import file_sha256
from scripts.prepare_multimodal72 import write_json
from scripts.run_aia72_canary import forward, state_digest
stage = root / 'data/raw/aia72_canary_v1_20261002'
output = root / 'outputs' / ('aia72_canary_notebook_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
print('Output:', output)
pd.set_option('display.max_colwidth', 65)

Output: /Users/mac/Documents/ChatGPT/Imaging_Project/gray-box-solar-flare-forecasting/outputs/aia72_canary_notebook_20261002T173917067308Z


### 2. Fresh temporal CNN–GRU
The executable architecture is reproduced here for inspection. Its classes are unchanged from the recorded upstream notebook; source provenance is in `results/aia72_canary_20261002/source_provenance.json`. It takes three frames × six channels and returns one 72-hour logit.

In [2]:
"""AIA CNN-GRU architecture reused from Watchman77/solar-flare-aia-training.

The three classes are preserved from 19A2_Temporal_AIA_CNN_GRU_Cycle24_Training.ipynb.
Only the module wrapper is new. No upstream weights or normalization are reused.
The new 72-hour label/role contract is supplied by this repository's loader.
Provenance: results/aia72_canary_20261002/source_provenance.json.
"""

import torch
from torch import nn

DROPOUT = 0.30

class ConvBlock(nn.Module):
    def __init__(self, cin, cout, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.GELU(),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.GELU(),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.net(x)

class FrameCNN(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(6, 32, 0.05),
            ConvBlock(32, 64, 0.05),
            ConvBlock(64, 128, 0.10),
            ConvBlock(128, 192, 0.10),
            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(192, embed_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        return self.proj(self.encoder(x))

class TemporalAIACNNGRU(nn.Module):
    def __init__(self, embed_dim=256, hidden_dim=192):
        super().__init__()
        self.frame_encoder = FrameCNN(embed_dim)
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        b,t,c,h,w = x.shape
        z = self.frame_encoder(x.reshape(b*t,c,h,w)).reshape(b,t,-1)
        _, hlast = self.gru(z)
        return self.head(hlast[-1]).squeeze(-1)


## Data
### 3. Check the selected cases and image bytes

In [3]:
root, stage, output = map(Path, [root, stage, output])
if output.exists():
    raise ValueError("Output exists; preserve prior checks")
source = json.loads((stage / "complete.json").read_text())
plan = json.loads((stage / "plan.json").read_text())
if source["plan_sha256"] != file_sha256(stage / "plan.json") or plan["cases_sha256"] != file_sha256(stage / "cases.csv.gz"):
    raise ValueError("Staging plan/cases changed")
cases = pd.read_csv(stage / "cases.csv.gz", low_memory=False)
records = {r["uri"]: r for r in source["objects"]}
required = set(cases[[f"history_uri_tminus{x}" for x in LAGS]].to_numpy().ravel())
if len(cases) != 14 or len(records) != len(source["objects"]) or set(records) != required:
    raise ValueError("Canary scope differs from its fixed selection")
output.mkdir(parents=True)
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
np.random.seed(17); torch.manual_seed(17)
started = time.monotonic()
image_checks = []
for uri, record in records.items():
    path = root / record["path_relative_to_repository"]
    if file_sha256(path) != record["sha256"]:
        raise ValueError("Pinned source content changed")
    validate_frame_identity(path, uri)
    raw, checked = load_aia_frame(path)
    image_checks.append({"uri": uri, "generation": record["generation"], "sha256": record["sha256"],
                         "metadata_schema": checked["metadata_schema"], "minimum": float(raw.min()),
                         "maximum": float(raw.max()), "shape": str(raw.shape), "finite": checked["finite"]})
checks = pd.DataFrame(image_checks)
checks.to_csv(output / "image_checks.csv", index=False)
display(cases.groupby('role', sort=False).size().rename('Checked cases').to_frame())
display(checks.groupby('metadata_schema').size().rename('Verified images').to_frame())

,Checked cases
role,
train,2
model_validation,2
probability_calibration,2
conformal_calibration,2
policy_validation,2
retrospective_cycle25,2
supplementary_2026,2


,Verified images
metadata_schema,
channel_names,33
numeric_wavelengths,9


### 4. Fit preprocessing on the two training cases only

In [4]:
normalization = fit_normalization(cases, records, root)
normalization["scope"] = "Technical canary training rows only; not full-experiment fitted statistics"
write_json(output / "normalization.json", normalization)
dataset = AIA72Dataset(cases, records, root, normalization)
loader = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=0)
train_cases = cases[cases.role.eq("train")].copy()
train_loader = DataLoader(AIA72Dataset(train_cases, records, root, normalization), batch_size=2, shuffle=False, num_workers=0)
model = TemporalAIACNNGRU()
initial_state = state_digest(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss()
training_rows = []
display(pd.DataFrame({'Channel': CHANNELS, 'Scale': normalization['channel_scale'], 'Mean': normalization['channel_mean'], 'Std': normalization['channel_std']}))

,Channel,Scale,Mean,Std
0,aia94,0.294464,0.718390,0.278235
1,aia131,0.382216,0.750405,0.205814
2,aia171,0.448832,0.743148,0.214079
3,aia193,0.469845,0.728952,0.237281
4,aia211,0.479597,0.733905,0.252993
5,aia335,0.500665,0.693334,0.254271


## Results
### 5. Two technical optimization steps
These steps test finite loss, gradients and parameter updates. They are not an estimate of forecast quality.

In [5]:
for step in range(2):
    model.train()
    images, labels, identities = next(iter(train_loader))
    if images.shape[1:] != (3, 6, 256, 256):
        raise ValueError("Unexpected AIA sequence shape")
    optimizer.zero_grad(set_to_none=True)
    loss = loss_fn(model(images), labels)
    if not torch.isfinite(loss):
        raise ValueError("Nonfinite technical training loss")
    loss.backward()
    grads = [p.grad for p in model.parameters() if p.grad is not None]
    if not grads or not all(torch.isfinite(g).all() for g in grads) or not any(g.abs().sum() > 0 for g in grads):
        raise ValueError("Missing, nonfinite or zero gradients")
    norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    training_rows.append({"technical_step": step + 1, "training_cases": len(labels), "bce_loss": float(loss.detach()),
                          "gradient_norm_before_clip": float(norm), "case_ids": "|".join(identities)})
pd.DataFrame(training_rows).to_csv(output / "technical_training_steps.csv", index=False)
fitted_state = state_digest(model)
if fitted_state == initial_state:
    raise ValueError("Training did not change model state")
display(pd.DataFrame(training_rows).drop(columns='case_ids'))

,technical_step,training_cases,bce_loss,gradient_norm_before_clip
0,1,2,0.752436,11.069344
1,2,2,0.278947,5.325875


### 6. Save and replay every selected case
Evaluation must leave weights and batch-normalization buffers unchanged. Reloaded logits must agree exactly with identical batching.

In [6]:
logits = forward(model, loader)
if state_digest(model) != fitted_state:
    raise ValueError("Evaluation changed model buffers or weights")
checkpoint = output / "technical_canary_only.pt"
torch.save(model.state_dict(), checkpoint)
replay = TemporalAIACNNGRU()
replay.load_state_dict(torch.load(checkpoint, weights_only=True, map_location="cpu"))
saved_norm = json.loads((output / "normalization.json").read_text())
replay_loader = DataLoader(AIA72Dataset(cases, records, root, saved_norm), batch_size=2, shuffle=False, num_workers=0)
replay_logits = forward(replay, replay_loader)
if not logits.equals(replay_logits) or state_digest(replay) != fitted_state:
    raise ValueError("Saved-model replay differs")
print('Saved-model replay: exact for all 14 cases; evaluation changed no model state.')

Saved-model replay: exact for all 14 cases; evaluation changed no model state.


### 7. Independently express the channel transformation
Compare the broadcast implementation against a channel-by-channel calculation on every checked object.

In [7]:
# Channelwise independent expression checks the broadcast transform on all
# checked objects. Resizing uses the same declared PyTorch interpolation API.
maximum_difference = 0.
for uri in records:
    raw, _ = load_aia_frame(root / records[uri]["path_relative_to_repository"])
    channels = []
    for channel in range(6):
        v = np.arcsinh(raw[:, :, channel] / np.float32(saved_norm["channel_scale"][channel]))
        channels.append((v - np.float32(saved_norm["channel_mean"][channel])) / np.float32(saved_norm["channel_std"][channel]))
    independent = torch.nn.functional.interpolate(torch.from_numpy(np.stack(channels))[None], size=(256, 256), mode="bilinear", align_corners=False)[0]
    actual = transform_image(raw, saved_norm)
    maximum_difference = max(maximum_difference, float((independent - actual).abs().max()))
    torch.testing.assert_close(independent, actual, atol=1e-6, rtol=1e-6)
logits.to_csv(output / "technical_logits_not_forecasts.csv", index=False)
print('Maximum independent transformation difference:', maximum_difference)

Maximum independent transformation difference: 0.0


## Takeaways
### 8. Save and summarize the executed technical evidence

In [8]:
result = {"status": "technical_aia72_canary_passed_not_scientific_training", "completed_utc": datetime.now(timezone.utc).isoformat(),
          "cases": len(cases), "roles": cases.role.value_counts().to_dict(), "objects_verified": len(records),
          "metadata_schemas": checks.metadata_schema.value_counts().to_dict(), "sequence_shape": [3, 6, 256, 256],
          "model_parameters": sum(p.numel() for p in model.parameters()), "technical_optimization_steps": 2,
          "preprocessing_fit_case_ids": normalization["training_case_ids"], "preprocessing_fit_objects": normalization["stats_files"],
          "source_48h_labels_used": False, "saved_model_replay": "exact_all_14_cases_same_batching",
          "evaluation_changed_weights_or_buffers": False, "independent_transform_max_absolute_difference": maximum_difference,
          "initial_state_sha256": initial_state, "final_state_sha256": fitted_state,
          "runtime": {"torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__, "device": "cpu", "threads": 2},
          "seconds": time.monotonic() - started, "full_72h_training_complete": False,
          "stage_receipt_sha256": file_sha256(stage / "complete.json"),
          "code_sha256": {name: file_sha256(root / "scripts" / name) for name in ["run_aia72_canary.py", "aia72_data.py", "aia72_model.py", "aia_io.py"]},
          "limitations": ["Fourteen boundary cases are a software check, not representative model evaluation",
                          "Two optimization steps use only two training-role cases; weights/statistics must not initialize the scientific run",
                          "Image identities/channel order/pixels checked; full archive quality and actual per-channel observation/delivery histories are not certified",
                          "GOES is not an input to this AIA-only canary"],
          "output_sha256": {p.name: file_sha256(p) for p in sorted(output.iterdir()) if p.is_file()}}
write_json(output / "summary.json", result)
display(pd.DataFrame([{'Cases': result['cases'], 'Verified images': result['objects_verified'], 'Parameters': result['model_parameters'], 'Technical steps': result['technical_optimization_steps'], 'Seconds': round(result['seconds'], 2), 'Full training complete': result['full_72h_training_complete']}]))
display(Markdown('**Passed: loader, two training steps and exact checkpoint replay. Full 72-hour training remains pending.**'))
print('Receipt:', output / 'summary.json')

,Cases,Verified images,Parameters,Technical steps,Seconds,Full training complete
0,14,42,1151233,2,10.89,False


**Passed: loader, two training steps and exact checkpoint replay. Full 72-hour training remains pending.**

Receipt: /Users/mac/Documents/ChatGPT/Imaging_Project/gray-box-solar-flare-forecasting/outputs/aia72_canary_notebook_20261002T173917067308Z/summary.json


The next experiment must use all frozen training cases, fresh full-training preprocessing and fresh weights. Benchmark the selected GPU before estimating time; save resumable checkpoints and preserve the reserved calibration and evaluation roles. Reusing the existing VM requires coordination with its active AIA work. No additional GPU resource was started by this notebook.